<a href="https://colab.research.google.com/github/zune2/colab-notebooks/blob/main/2026-11-21-Qwen2.5-7B-Instruct-Cross-Encoder-LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q -U bitsandbytes onnx onnxruntime onnxscript

In [4]:
# Cell2
import gc
import json
import re
import random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. 도구 스키마 및 파라미터 명세 정의
TOOLS_SPEC = {
    "get_weather": {
        "desc": "Tool: get_weather | Desc: 특정 지역의 날짜별 날씨와 기온, 강수 확률을 조회합니다.",
        "schema": {
            "name": "get_weather",
            "parameters": {
                "location": "도시 또는 동네 이름 (예: 서울, 분당 정자동, 부산)",
                "date": "날짜 표현 (예: 오늘, 내일, 모레, 주말)"
            }
        }
    },
    "search_order": {
        "desc": "Tool: search_order | Desc: 사용자가 구매한 상품의 주문 내역, 배송 상태, 택배 위치를 조회합니다.",
        "schema": {
            "name": "search_order",
            "parameters": {
                "item_name": "구매한 상품명 (예: 운동화, 노트북, 책, 없으면 null)",
                "order_id": "주문번호 숫자 (없으면 null)"
            }
        }
    },
    "cancel_subscription": {
        "desc": "Tool: cancel_subscription | Desc: 유료 멤버십이나 정기 구독 서비스를 해지하고 결제를 취소합니다.",
        "schema": {
            "name": "cancel_subscription",
            "parameters": {
                "plan_name": "구독 또는 멤버십 이름 (예: 유료 멤버십, 프리미엄, 없으면 '전체')",
                "immediate": "즉시 해지 여부 (true 또는 false)"
            }
        }
    }
}

LLM_ID = "Qwen/Qwen2.5-7B-Instruct"

# 이미 메모리에 로드되어 있으면 재로드 생략
if "llm_model" not in globals():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    print("⏳ 1/3: Qwen2.5-7B-Instruct (4-bit) 로딩 중...")
    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_ID, quantization_config=bnb_config, device_map="auto"
    )

def generate_tool_data_with_args(tool_name: str, spec: dict, target_count: int = 50) -> list[dict]:
    collected = []
    seen_queries = set()
    batch_num = 1

    while len(collected) < target_count:
        schema_json = json.dumps(spec["schema"], ensure_ascii=False)
        prompt = (
            f"당신은 한국어 Function Calling 학습 데이터 생성기입니다.\n"
            f"아래 도구 스키마에 맞는 자연스러운 한국어 사용자 질문과 정답 인자(args) 20세트를 생성하세요.\n\n"
            f"[도구 설명]\n{spec['desc']}\n"
            f"[파라미터 구조]\n{schema_json}\n\n"
            f"[생성 규칙]\n"
            f"1. 반말, 존댓말, 구어체, 의문문 등 다양한 한국어 표현을 사용하세요.\n"
            f"2. 반드시 JSON 배열(Array of Objects) 형식으로만 출력하세요.\n"
            f"출력 형식 예시:\n"
            f"[{{\"query\": \"내일 판교 비 와?\", \"tool\": \"{tool_name}\", \"args\": {{\"location\": \"판교\", \"date\": \"내일\"}}}}]"
        )
        messages = [{"role": "user", "content": prompt}]
        text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = llm_tokenizer([text], return_tensors="pt").to("cuda")
        input_len = inputs.input_ids.shape[1]

        with torch.inference_mode():
            outputs = llm_model.generate(
                **inputs, max_new_tokens=900, temperature=0.85, top_p=0.9, do_sample=True
            )

        gen_text = llm_tokenizer.decode(outputs[0, input_len:], skip_special_tokens=True)
        match = re.search(r"\[.*\]", gen_text, re.DOTALL)
        if match:
            try:
                items = json.loads(match.group())
                for item in items:
                    q = item.get("query", "").strip()
                    args = item.get("args", {})
                    if q and q not in seen_queries and isinstance(args, dict):
                        seen_queries.add(q)
                        collected.append({"query": q, "tool": tool_name, "args": args})
            except json.JSONDecodeError:
                pass
        print(f"   [{tool_name}] Batch {batch_num} 완료 -> 누적 {len(collected)}/{target_count}개")
        batch_num += 1
        if batch_num > 5:
            break

    return collected[:target_count]

def generate_general_chat(target_count: int = 50) -> list[dict]:
    collected = []
    seen = set()
    for _ in range(3):
        prompt = (
            "날씨, 주문/배송, 구독 해지와 전혀 관련 없는 일반 일상 대화, 인사, 맛집/영화 추천, 코딩 질문 25개를 "
            "JSON 문자열 배열([\"질문1\", \"질문2\"])로만 출력하세요."
        )
        messages = [{"role": "user", "content": prompt}]
        text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = llm_tokenizer([text], return_tensors="pt").to("cuda")
        input_len = inputs.input_ids.shape[1]

        with torch.inference_mode():
            outputs = llm_model.generate(**inputs, max_new_tokens=500, temperature=0.85, do_sample=True)

        gen_text = llm_tokenizer.decode(outputs[0, input_len:], skip_special_tokens=True)
        match = re.search(r"\[.*\]", gen_text, re.DOTALL)
        if match:
            try:
                for q in json.loads(match.group()):
                    if isinstance(q, str) and q.strip() not in seen:
                        seen.add(q.strip())
                        collected.append({"query": q.strip(), "tool": None, "args": {}})
            except Exception:
                pass
    return collected[:target_count]

print("🚀 2/3: (질문 + 함수명 + 정답 인자) 합성 데이터 생성 시작...")
synthetic_dataset = []
for t_name, t_spec in TOOLS_SPEC.items():
    synthetic_dataset.extend(generate_tool_data_with_args(t_name, t_spec, target_count=50))
synthetic_dataset.extend(generate_general_chat(target_count=50))

# GPU 메모리 정리
del llm_model
del llm_tokenizer
gc.collect()
torch.cuda.empty_cache()

print(f"\n✅ 3/3: 총 {len(synthetic_dataset)}개 합성 데이터(질문+함수+인자) 생성 완료!")
print("샘플 3건 미리보기:")
for sample in random.sample(synthetic_dataset, 3):
    print(" ->", sample)


⏳ 1/3: Qwen2.5-7B-Instruct (4-bit) 로딩 중...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

🚀 2/3: (질문 + 함수명 + 정답 인자) 합성 데이터 생성 시작...
   [get_weather] Batch 1 완료 -> 누적 17/50개
   [get_weather] Batch 2 완료 -> 누적 36/50개
   [get_weather] Batch 3 완료 -> 누적 55/50개
   [search_order] Batch 1 완료 -> 누적 0/50개
   [search_order] Batch 2 완료 -> 누적 0/50개
   [search_order] Batch 3 완료 -> 누적 0/50개
   [search_order] Batch 4 완료 -> 누적 16/50개
   [search_order] Batch 5 완료 -> 누적 16/50개
   [cancel_subscription] Batch 1 완료 -> 누적 0/50개
   [cancel_subscription] Batch 2 완료 -> 누적 0/50개
   [cancel_subscription] Batch 3 완료 -> 누적 15/50개
   [cancel_subscription] Batch 4 완료 -> 누적 15/50개
   [cancel_subscription] Batch 5 완료 -> 누적 15/50개

✅ 3/3: 총 131개 합성 데이터(질문+함수+인자) 생성 완료!
샘플 3건 미리보기:
 -> {'query': '주말 분당 정자동 비 오나?', 'tool': 'get_weather', 'args': {'location': '분당 정자동', 'date': '주말'}}
 -> {'query': '오늘 경기도 남양주에서 얼마나 더울까요?', 'tool': 'get_weather', 'args': {'location': '남양주', 'date': '오늘'}}
 -> {'query': '유료 멤버십 해지요청이 가능한가요?', 'tool': 'cancel_subscription', 'args': {'plan_name': '유료 멤버십', 'immediate': 'false'}}


In [5]:
# Cell 3
import os
import random
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

random.seed(42)
random.shuffle(synthetic_dataset)

# All-Pairs 매칭 (정답 도구=1.0, 오답 도구=0.0)
all_pairs = []
for item in synthetic_dataset:
    q = item["query"]
    target_tool = item["tool"]
    for tool_name, spec in TOOLS_SPEC.items():
        score = 1.0 if tool_name == target_tool else 0.0
        all_pairs.append((q, spec["desc"], score))

dataset_dict = {
    "query": [q for q, t_desc, l in all_pairs],
    "tool_text": [t_desc for q, t_desc, l in all_pairs],
    "label": [l for q, t_desc, l in all_pairs]
}
dataset = Dataset.from_dict(dataset_dict).train_test_split(test_size=0.1, seed=42)

MODEL_ID = "klue/roberta-base"
router_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
router_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=1)

def preprocess_function(examples):
    return router_tokenizer(
        examples["query"],
        examples["tool_text"],
        truncation=True,
        max_length=256,
        padding="max_length",
        return_token_type_ids=False
    )

encoded_dataset = dataset.map(preprocess_function, batched=True)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    warmup_steps=20,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=30,
    fp16=False,
    report_to="none"
)

trainer = Trainer(
    model=router_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
)

print("🚀 Cross-Encoder 라우터 파인튜닝 시작...")
trainer.train()
print("✅ 파인튜닝 완료!")


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

... (4 outputs left out)

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

🚀 Cross-Encoder 라우터 파인튜닝 시작...


Epoch,Training Loss,Validation Loss
1,0.195730,0.080671
2,0.028082,0.003574
3,0.008902,0.003141
4,0.008061,0.002145
5,0.004865,0.001186
6,0.003958,0.000685
7,0.003535,0.000439
8,0.002919,0.000384


✅ 파인튜닝 완료!


In [6]:
# Cell 4
ONNX_DIR = "./onnx_router"
os.makedirs(ONNX_DIR, exist_ok=True)

router_model.eval().cpu()

dummy_input = router_tokenizer(
    "테스트 질문", "Tool: test | Desc: 테스트",
    return_tensors="pt", padding="max_length", max_length=256,
    return_token_type_ids=False
)

onnx_path = os.path.join(ONNX_DIR, "model.onnx")

torch.onnx.export(
    router_model,
    (dummy_input["input_ids"], dummy_input["attention_mask"]),
    onnx_path,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"}
    },
    opset_version=14,
    dynamo=False
)

router_tokenizer.save_pretrained(ONNX_DIR)
!zip -r onnx_router.zip {ONNX_DIR}
print("✅ ONNX 변환 완료!")


/tmp/ipykernel_2740/3004769071.py:15: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:208: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.13/dist-packages/transformers/integrations/sdpa_attention.py:140: TracerWarning: Converting a tensor to a Python boolea

  adding: onnx_router/ (stored 0%)
  adding: onnx_router/tokenizer_config.json (deflated 50%)
  adding: onnx_router/model.onnx (deflated 13%)
  adding: onnx_router/tokenizer.json (deflated 69%)
✅ ONNX 변환 완료!


In [9]:
# [Cell 5] End-to-End 벤치마크 채점 전체 코드 (단독 실행 가능)
import os
import json
import re
import time
import torch
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. 도구 스키마 정의
TOOLS_SPEC = {
    "get_weather": {
        "desc": "Tool: get_weather | Desc: 특정 지역의 날짜별 날씨와 기온, 강수 확률을 조회합니다.",
        "schema": {
            "name": "get_weather",
            "parameters": {
                "location": "도시 또는 동네 이름 (예: 서울, 분당 정자동, 부산)",
                "date": "날짜 표현 (예: 오늘, 내일, 모레, 주말)"
            }
        }
    },
    "search_order": {
        "desc": "Tool: search_order | Desc: 사용자가 구매한 상품의 주문 내역, 배송 상태, 택배 위치를 조회합니다.",
        "schema": {
            "name": "search_order",
            "parameters": {
                "item_name": "구매한 상품명 (예: 운동화, 노트북, 책, 없으면 null)",
                "order_id": "주문번호 숫자 (없으면 null)"
            }
        }
    },
    "cancel_subscription": {
        "desc": "Tool: cancel_subscription | Desc: 유료 멤버십이나 정기 구독 서비스를 해지하고 결제를 취소합니다.",
        "schema": {
            "name": "cancel_subscription",
            "parameters": {
                "plan_name": "구독 또는 멤버십 이름 (예: 유료 멤버십, 프리미엄, 없으면 '전체')",
                "immediate": "즉시 해지 여부 (true 또는 false)"
            }
        }
    }
}

# 2. ONNX 라우터 및 토크나이저 로드
ONNX_DIR = "./onnx_router"
router_tokenizer = AutoTokenizer.from_pretrained(ONNX_DIR)
session = ort.InferenceSession(f"{ONNX_DIR}/model.onnx", providers=["CPUExecutionProvider"])

# 3. 인자 추출용 Local LLM 로드 (이미 메모리에 로드되어 있으면 재로드 생략)
LLM_ID = "Qwen/Qwen2.5-7B-Instruct"
if "llm_model" not in globals() or llm_model is None:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    print("⏳ 인자 추출용 Qwen2.5-7B-Instruct (4-bit) 로딩 중...")
    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_ID, quantization_config=bnb_config, device_map="auto"
    )

# 4. 평가용 테스트케이스 (함수명 + 정답 인자 포함)
eval_testcases = [
    {"query": "정자동 지금 소나기 내리나?", "tool": "get_weather", "args": {"location": "정자동", "date": "지금"}},
    {"query": "내일 서울 기온 몇 도까지 떨어져?", "tool": "get_weather", "args": {"location": "서울", "date": "내일"}},
    {"query": "주말에 부산 비 올 확률 알려줘", "tool": "get_weather", "args": {"location": "부산", "date": "주말"}},
    {"query": "모레 제주도 날씨 맑은지 봐줘", "tool": "get_weather", "args": {"location": "제주도", "date": "모레"}},

    {"query": "어제 시킨 운동화 배송 어디쯤 왔어?", "tool": "search_order", "args": {"item_name": "운동화", "order_id": None}},
    {"query": "주문번호 987654 배송 상태 조회해 줘", "tool": "search_order", "args": {"item_name": None, "order_id": "987654"}},
    {"query": "지난주에 결제한 책 출발했나요?", "tool": "search_order", "args": {"item_name": "책", "order_id": None}},
    {"query": "방금 주문한 노트북 결제 내역 확인 좀", "tool": "search_order", "args": {"item_name": "노트북", "order_id": None}},

    {"query": "유료 멤버십 오늘 바로 해지해 주세요", "tool": "cancel_subscription", "args": {"plan_name": "유료 멤버십", "immediate": True}},
    {"query": "다음 달부터 프리미엄 이용권 정기결제 취소할래", "tool": "cancel_subscription", "args": {"plan_name": "프리미엄 이용권", "immediate": False}},
    {"query": "구독 서비스 탈퇴 처리 부탁합니다", "tool": "cancel_subscription", "args": {"plan_name": "전체", "immediate": False}},

    {"query": "오늘 점심 메뉴 뭐 먹을까?", "tool": None, "args": {}},
    {"query": "파이썬 딕셔너리 정렬하는 법 알려줘", "tool": None, "args": {}},
    {"query": "고마워 덕분에 잘 해결됐어!", "tool": None, "args": {}},
]

# 5. Local LLM 인자(JSON) 추출 함수
def extract_arguments(query: str, tool_name: str) -> dict:
    params_spec = json.dumps(TOOLS_SPEC[tool_name]["schema"]["parameters"], ensure_ascii=False)
    prompt = (
        f"사용자 질문에서 아래 파라미터에 해당하는 값을 추출하여 순수 JSON 객체로만 출력하세요.\n"
        f"'name'이나 'parameters' 같은 상위 키로 감싸지 말고, 파라미터 키만 최상위에 두세요.\n"
        f"값이 없으면 null을 넣고, boolean은 true/false로 출력하세요.\n\n"
        f"[파라미터 명세]\n{params_spec}\n"
        f"[출력 예시]\n{{\"location\": \"서울\", \"date\": \"내일\"}}\n\n"
        f"사용자 질문: {query}"
    )
    messages = [{"role": "user", "content": prompt}]
    text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer([text], return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].size(1)  # 정확한 정수 토큰 길이 추출

    with torch.inference_mode():
        outputs = llm_model.generate(**inputs, max_new_tokens=120, temperature=0.05, do_sample=True)

    gen_text = llm_tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    match = re.search(r"\{.*\}", gen_text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            if "parameters" in parsed and isinstance(parsed["parameters"], dict):
                return parsed["parameters"]
            return parsed
        except Exception:
            return {}
    return {}

# 6. 인자 일치 여부 검증 함수
def check_args_match(expected: dict, predicted: dict) -> bool:
    for k, v in expected.items():
        pred_v = predicted.get(k)
        if v in [None, "전체"] and pred_v in [None, "", "null", "None", "전체"]:
            continue
        if isinstance(v, bool):
            if v is False and pred_v in [False, "false", "False", None, "null"]:
                continue
            if v is True and pred_v in [True, "true", "True"]:
                continue
            return False
        exp_str = str(v).replace(" ", "").lower()
        pred_str = str(pred_v).replace(" ", "").lower()
        if exp_str == "지금" and pred_str in ["지금", "오늘", "현재"]:
            continue
        if exp_str not in pred_str and pred_str not in exp_str:
            return False
    return True

# 7. 전체 테스트케이스 실행 및 채점
tool_names = list(TOOLS_SPEC.keys())
tool_descs = [TOOLS_SPEC[k]["desc"] for k in tool_names]
THRESHOLD = 0.55

tool_correct = 0
e2e_correct = 0
router_latencies = []

print("🔍 End-to-End 채점 결과 (1단계: ONNX 함수 라우팅 -> 2단계: LLM 인자 추출)")
print("=" * 95)

for idx, tc in enumerate(eval_testcases, 1):
    q = tc["query"]
    exp_tool = tc["tool"]
    exp_args = tc["args"]

    t0 = time.perf_counter()
    inputs = router_tokenizer(
        [q] * len(tool_descs), tool_descs,
        padding=True, truncation=True, max_length=256, return_tensors="np",
        return_token_type_ids=False
    )
    raw_logits, = session.run(["logits"], {
        "input_ids": inputs["input_ids"].astype(np.int64),
        "attention_mask": inputs["attention_mask"].astype(np.int64)
    })
    scores = np.clip(raw_logits.squeeze(-1), 0.0, 1.0)
    router_latencies.append((time.perf_counter() - t0) * 1000)

    score_map = {name: float(s) for name, s in zip(tool_names, scores)}
    best_tool = max(score_map, key=score_map.get)
    pred_tool = best_tool if score_map[best_tool] >= THRESHOLD else None

    pred_args = extract_arguments(q, pred_tool) if pred_tool else {}

    is_tool_ok = (pred_tool == exp_tool)
    is_args_ok = check_args_match(exp_args, pred_args) if is_tool_ok else False

    if is_tool_ok:
        tool_correct += 1
    if is_tool_ok and is_args_ok:
        e2e_correct += 1

    mark = "✅ O" if (is_tool_ok and is_args_ok) else ("⚠️ 함수만 일치" if is_tool_ok else "❌ X")
    print(f"[{idx:02d}] {mark} | 질문: {q}")
    print(f"      정답: {exp_tool} {exp_args}")
    print(f"      예측: {pred_tool} {pred_args}")

total = len(eval_testcases)
print("\n" + "=" * 95)
print(f"📊 [최종 End-to-End 벤치마크 리포트]")
print(f" - 1단계 함수 라우팅 정확도 (Tool Accuracy) : {(tool_correct/total)*100:.2f}% ({tool_correct}/{total})")
print(f" - 2단계 함수+인자 최종 정확도 (E2E Accuracy): {(e2e_correct/total)*100:.2f}% ({e2e_correct}/{total})")
print(f" - 평균 ONNX 라우터 추론 속도 (Latency)    : {np.mean(router_latencies):.2f} ms")
print("=" * 95)


🔍 End-to-End 채점 결과 (1단계: ONNX 함수 라우팅 -> 2단계: LLM 인자 추출)
[01] ⚠️ 함수만 일치 | 질문: 정자동 지금 소나기 내리나?
      정답: get_weather {'location': '정자동', 'date': '지금'}
      예측: get_weather {'location': '분당 정자동', 'date': None}
[02] ✅ O | 질문: 내일 서울 기온 몇 도까지 떨어져?
      정답: get_weather {'location': '서울', 'date': '내일'}
      예측: get_weather {'location': '서울', 'date': '내일'}
[03] ✅ O | 질문: 주말에 부산 비 올 확률 알려줘
      정답: get_weather {'location': '부산', 'date': '주말'}
      예측: get_weather {'location': '부산', 'date': '주말'}
[04] ✅ O | 질문: 모레 제주도 날씨 맑은지 봐줘
      정답: get_weather {'location': '제주도', 'date': '모레'}
      예측: get_weather {'location': '제주도', 'date': '모레'}
[05] ✅ O | 질문: 어제 시킨 운동화 배송 어디쯤 왔어?
      정답: search_order {'item_name': '운동화', 'order_id': None}
      예측: search_order {'item_name': '운동화', 'order_id': None}
[06] ⚠️ 함수만 일치 | 질문: 주문번호 987654 배송 상태 조회해 줘
      정답: search_order {'item_name': None, 'order_id': '987654'}
      예측: search_order {}
[07] ❌ X | 질문: 지난주에 결제한 책 출발했나요?
      정답: search_order {'item_n

# Lightweight Semantic Tool Router & Function Calling Pipeline

저사양 CPU 서버(가용 RAM 1.4GB 이하) 환경에서 초고속으로 사용자 의도를 분류하고 외부 도구(Function/API)를 호출하기 위해 구축한 **ONNX Cross-Encoder 기반 Semantic Tool Router** 프로젝트입니다.

---

## 1. 프로젝트 개요 (Overview)

* **목적:** 무거운 Local LLM을 상시 구동할 수 없는 저사양 서버 환경에서, 경량 Encoder 모델(`klue/roberta-base`)로 **호출할 도구(Tool)만 초고속 필터링**한 뒤 필요한 스키마만 LLM에 전달하여 토큰 비용과 지연 시간(Latency)을 최소화합니다.
* **핵심 특징:**
  * **Zero-API 합성 데이터 구축:** Google Colab(T4 GPU) 환경에서 `Qwen2.5-7B-Instruct (4-bit)`를 활용해 `(질문 + 정답 함수명 + 정답 인자 JSON)` 데이터셋을 자동 생성합니다.
  * **All-Pairs Cross-Encoder 학습:** 모든 질문과 전체 도구 스키마를 1:N으로 페어링하여 오답 도구(Hard Negative)에 대한 과적합을 차단합니다.
  * **ONNX Runtime 경량 추론:** 추론 서버에서 무거운 PyTorch 없이 `onnxruntime` 단독 실행으로 메모리 점유율을 최소화합니다.

---

## 2. 벤치마크 성능 요약 (Benchmark Results)

학습 데이터에 포함되지 않은 **미학습(Unseen) 14개 평가 문항** 기준 정량 평가 결과입니다.

| 평가 지표 (Metric) | 측정 결과 | 비고 |
| :--- | :---: | :--- |
| **1단계 함수 라우팅 정확도 (Tool Accuracy)** | **92.86%** (13/14) | Threshold `0.55` 기준 |
| **2단계 함수+인자 최종 정확도 (E2E Accuracy)** | **71.43%** (10/14) | 4-bit 경량 LLM 인자 추출 결합 기준 |
| **평균 라우팅 추론 속도 (Router Latency)** | **247.76 ms** | CPU 단일 스레드 기준 배치 추론 |
| **파인튜닝 최종 검증 손실 (Validation Loss)** | **0.000384** | `klue/roberta-base`, 8 Epochs 수렴 |

> 상세한 시스템 설계 구조와 트러블슈팅 내역은 [ARCHITECTURE.md](./ARCHITECTURE.md)를 참고하세요.

---

## 3. 디렉토리 구조 (Directory Structure)

```text
modernBERT_router/
├── README.md                  # 프로젝트 소개, 벤치마크 및 실행 가이드
├── ARCHITECTURE.md            # 시스템 아키텍처, 학습 파이프라인 및 트러블슈팅 상세 문서
├── train_and_eval.ipynb       # Colab용 합성 데이터 생성, 파인튜닝, ONNX 변환 및 E2E 채점 노트북
├── synthetic_dataset.json     # Local LLM으로 생성된 (질문+함수+인자) 합성 데이터셋
├── run_onnx_router.py         # Linux 서버용 초경량 ONNX 라우터 추론 스크립트
└── onnx_router/               # ONNX 변환 완료된 모델 및 토크나이저 파일
    ├── model.onnx
    ├── tokenizer.json
    └── tokenizer_config.json
```

---

## 4. 시작하기 (Quick Start)

### Step 1: 모델 학습 및 ONNX 변환 (Google Colab T4 GPU)
저사양 서버의 메모리 부족(OOM)을 방지하기 위해 데이터 생성과 학습은 GPU 환경에서 수행합니다.
1. `train_and_eval.ipynb`를 열고 런타임 유형을 **T4 GPU**로 설정합니다.
2. `[Cell 1] ~ [Cell 4]`를 순서대로 실행하여 합성 데이터 생성, Cross-Encoder 파인튜닝, ONNX 변환을 진행합니다.
3. 생성된 `onnx_router.zip`을 다운로드하여 추론 서버로 업로드합니다.

### Step 2: 추론 서버 환경 설치 (Linux CPU)
추론 서버에는 PyTorch를 설치하지 않고 가벼운 런타임만 설치합니다.

```bash
python3 -m venv router_env
source router_env/bin/activate

pip install --no-cache-dir numpy transformers onnxruntime
unzip -o onnx_router.zip
```

### Step 3: 라우터 실행
```bash
python run_onnx_router.py
```

---

## 5. 향후 개선 계획 (Roadmap)

* **인자 추출 프롬프트 고도화:** 도구 스키마 내 Few-shot JSON 예시를 추가하여 2단계 E2E 정확도를 95% 이상으로 상향
* **경계선 데이터 보강:** `"결제한 상품 출발 여부"` 등 유사 키워드가 겹치는 문장 패턴을 합성 데이터에 추가하여 1단계 라우팅 정확도 개선
* **API 서빙화:** FastAPI 기반 비동기 REST API 서버로 래핑하여 외부 에이전트 시스템과 연동


# System Architecture & Technical Design

이 문서는 **Semantic Tool Router & Function Calling Pipeline**의 전체 아키텍처, 데이터 생성 및 학습 파이프라인, 그리고 개발 과정에서 해결한 주요 기술적 이슈(Troubleshooting)를 기록한 설계 문서입니다.

---

## 1. 시스템 아키텍처 (End-to-End Architecture)

전체 파이프라인은 **① 오프라인 학습 파이프라인(GPU)**과 **② 온라인 추론 파이프라인(CPU)**으로 분리되어 동작합니다.

```text
====================================================================================
 [Phase 1: Offline Training & Export Pipeline (Google Colab T4 GPU)]
====================================================================================
  [도구 스키마 명세] ──▶ [Qwen2.5-7B (4-bit)] ──▶ 합성 데이터 (질문 + 함수명 + 인자 JSON)
                                                         │
                                                         ▼  All-Pairs 페어링 (1:N 매칭)
  [onnx_router.zip] ◀── [ONNX Export (Opset 14)] ◀── [klue/roberta-base 파인튜닝]


====================================================================================
 [Phase 2: Online Inference Pipeline (Linux CPU Server, RAM 1.4GB)]
====================================================================================
  [사용자 쿼리] ("어제 시킨 운동화 배송 어디쯤 왔어?")
        │
        ▼  Tokenizer (Batch Pairing: Query × Registered Tools)
  [ONNX Cross-Encoder 추론] (CPUExecutionProvider, ~240ms)
        │
        ├─▶ Score < 0.55 (모든 도구) ──▶ [General Chat] LLM 직접 응답
        │
        └─▶ Score >= 0.55 (도구 선택) ──▶ 선택된 1개 도구 스키마만 LLM에 전달
                                                 │
                                                 ▼
                                    [JSON 인자 추출] {"item_name": "운동화", "order_id": null}
                                                 │
                                                 ▼
                                    [실제 Python 함수 실행] search_order(**args)
```

---

## 2. 핵심 설계 결정 (Key Design Decisions)

### ① Bi-Encoder 대신 Cross-Encoder 채택
* **이유:** 사용자 발화(*"분당 지금 우산 챙겨야 해?"*)와 도구 설명(*"Tool: get_weather | Desc: 특정 지역의..."*)은 길이와 문체가 다른 **비대칭(Asymmetric) 텍스트**입니다. Bi-Encoder(단순 코사인 유사도) 방식에서는 모든 점수가 `0.73~0.85` 구간에 뭉개지는 현상이 발생했으나, 두 문장을 `[CLS] Query [SEP] Tool_Desc [SEP]`로 결합해 Cross-Attention을 수행하는 Cross-Encoder 도입 후 `1.00` vs `0.00`으로 완벽히 분리되었습니다.

### ② All-Pairs (전체 조합) 학습 데이터 구성
* **이유:** 특정 질문에 대해 정답 도구(`1.0`)만 학습시키면 모델이 요청형 어미(`~해줘`, `~확인 좀`)만 보고 모든 도구에 `1.0`을 출력하는 **Shortcut Learning(과적합)**이 발생합니다.
* **해결:** 모든 생성 질문을 등록된 전체 도구와 1:N으로 교차 매칭하여 **정답 도구 1개에는 `1.0`**, **나머지 모든 오답 도구에는 `0.0`**을 부여하는 All-Pairs 데이터셋을 구축했습니다.

### ③ 독립 스코어링 (Multi-Tool Routing 지원)
* Softmax 대신 단일 스칼라 로짓(`num_labels=1`)을 출력하고 `[0.0, 1.0]`으로 클리핑(`np.clip`)하여 각 도구별 독립 확률을 산출합니다. 이를 통해 복합 질문 입력 시 임계값(`0.55`)을 넘는 여러 도구를 동시에 호출할 수 있습니다.

### ④ 스키마 필터링을 통한 인자 추출 최적화
* 라우터가 선택한 **단 1개의 도구 파라미터 명세(`schema["parameters"]`)**만 후속 LLM에 전달합니다. 전체 도구 목록을 전달할 때보다 프롬프트 토큰을 대폭 절감하며, 경량 4-bit LLM에서도 JSON 파싱 오류와 중첩 래핑(`{"parameters": {...}}`)을 방지합니다.

---

## 3. 파인튜닝 명세 (Training Specifications)

* **합성 데이터 규모:** 클래스당 50문항 × 4개 클래스(`get_weather`, `search_order`, `cancel_subscription`, `None`) = 200문항 → All-Pairs 변환 후 **600개 학습 쌍** (Train 90% / Eval 10%)
* **백본 모델:** `klue/roberta-base`
* **토크나이저 설정:** `max_length=256`, `padding="max_length"`, **`return_token_type_ids=False`**
* **학습 하이퍼파라미터:**
  * Loss Function: `MSELoss` (Float 라벨 `1.0` / `0.0` 회귀 수렴)
  * Learning Rate: `3e-5` (`warmup_steps=20`, `weight_decay=0.01`)
  * Batch Size: `8` / Epochs: `8` / `fp16=False`

---

## 4. 트러블슈팅 기록 (Troubleshooting Log)

| 이슈 분류 | 현상 및 에러 로그 | 근본 원인 | 해결 방법 |
| :--- | :--- | :--- | :--- |
| **백본 모델 한계** | `SimCSE` 임베딩 모델 및 INT8 양자화 시 점수가 `0.70~0.73`으로 압축됨 | 임베딩 공간이 고정된 백본 특성 및 양자화 활성화 값 범위 손실 | MLM 기반 `klue/roberta-base`로 변경 및 FP32 ONNX 포맷 유지 |
| **CUDA 런타임 에러** | `CUDA error: device-side assert triggered` 발생 | 문장 쌍 토큰화 시 생성된 `token_type_ids=1`이 RoBERTa의 `type_vocab_size(1)` 초과 | 토크나이징 시 `return_token_type_ids=False` 옵션 추가 |
| **점수 스케일 왜곡** | 정답 로짓이 `0.738`, 오답이 `0.498`로 출력됨 | `MSELoss`로 `1.0/0.0`에 수렴한 출력값에 `Sigmoid` 함수를 중복 적용 | `Sigmoid` 제거 후 `np.clip(logits, 0.0, 1.0)`으로 직접 변환 |
| **ONNX 내보내기 충돌** | `optimum` 설치 시 `huggingface-hub` 버전 충돌 및 `onnxscript` 누락 | Colab 내장 패키지와 `optimum` 간 의존성 불일치 | `optimum` 제거 후 순수 `torch.onnx.export(..., dynamo=False)` 사용 |
| **JSON 중첩 래핑** | LLM이 `{"name": ..., "parameters": {...}}` 형태로 감싸서 출력 | 프롬프트에 최상위 함수 스키마 전체를 전달함 | `schema["parameters"]`만 전달하고 출력 시 자동 언래핑 로직 추가 |
